In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

: 

In [ ]:
class_param = pd.read_csv('../../data/classification_parameters.csv')
reservoir_info = pd.read_csv('../../data/reservoir_info.csv')
well_data = pd.read_csv('../../data/spe_africa_dseats_datathon_2025_wells_dataset.csv')
sample_sub = pd.read_csv('../../data/Innovisors_DSEATS_Africa_2025_Classification.csv')

In [ ]:
sample_sub.head(4)

In [ ]:
class_param

In [ ]:
reservoir_info

In [ ]:
reservoir_info.info()

In [ ]:
cols = reservoir_info.columns.drop('Reservoir Name').to_list()
cols

In [ ]:
# Function to clean numeric columns (handles commas, spaces, etc.)
def clean_numeric_column(series):
    return (series.astype(str)
            .str.replace(',', '')  # Remove commas
            .str.replace(' ', '')   # Remove spaces
            .str.strip()           # Remove leading/trailing whitespace
    )

In [ ]:
# Apply to production columns
for col in cols:
    reservoir_info[col] = pd.to_numeric(clean_numeric_column(reservoir_info[col]), errors='coerce')

In [ ]:
# Check data types after conversion
print("Data types after conversion:")
print(reservoir_info.dtypes)

In [ ]:
well_data.info()

In [ ]:
well_data.describe(include='all')

In [ ]:
# Check for non-numeric patterns in pressure columns
pressure_cols = ['BOTTOMHOLE_FLOWING_PRESSURE (PSI)', 'ANNULUS_PRESS (PSI)', 'WELL_HEAD_PRESSURE (PSI)']

# Check cumulative production columns
prod_cols = ['CUMULATIVE_OIL_PROD (STB)', 'CUMULATIVE_FORMATION_GAS_PROD (MSCF)', 
             'CUMULATIVE_TOTAL_GAS_PROD (MSCF)', 'CUMULATIVE_WATER_PROD (BBL)']

# Function to clean numeric columns (handles commas, spaces, etc.)
def clean_numeric_column(series):
    return (series.astype(str)
            .str.replace(',', '')  # Remove commas
            .str.replace(' ', '')   # Remove spaces
            .str.strip()           # Remove leading/trailing whitespace
    )

In [ ]:
# Apply to production columns
for col in prod_cols:
    well_data[col] = pd.to_numeric(clean_numeric_column(well_data[col]), errors='coerce')

In [ ]:
# Apply to pressure columns
for col in pressure_cols:
    well_data[col] = pd.to_numeric(clean_numeric_column(well_data[col]), errors='coerce')

In [ ]:
# Convert date column (adjust format based on what you see)
well_data['PROD_DATE'] = pd.to_datetime(well_data['PROD_DATE'], errors='coerce')

In [ ]:
# Check data types after conversion
print("Data types after conversion:")
print(well_data.dtypes)

In [ ]:
# Validate realistic ranges
print("\nPressure range:")
for col in pressure_cols:
    print(f"{col}: {well_data[col].min():.1f} - {well_data[col].max():.1f}")

print("\nProduction ranges:")
for col in prod_cols:
    print(f"{col}: {well_data[col].min():.1f} - {well_data[col].max():.1f}")

In [ ]:
well_data.head(4)

In [ ]:
missing_count = well_data.isnull().sum()
missing_count

In [ ]:
well_data.describe(include="all")

In [ ]:
well_data['DAY'] = well_data['PROD_DATE'].dt.day_name()
well_data['MONTH'] = well_data['PROD_DATE'].dt.month_name()
well_data['Year'] = well_data['PROD_DATE'].dt.year
well_data.head(5)

In [ ]:
# Print min and max dates
print("Min date:", well_data['PROD_DATE'].min())
print("Max date:", well_data['PROD_DATE'].max())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Count number of records per well
well_counts = well_data['WELL_NAME'].value_counts().sort_values(ascending=False)

# Plot
plt.figure(figsize=(12, 6))
sns.barplot(x=well_counts.index, y=well_counts.values, palette='Blues_r')

# Labels and title
plt.xlabel('Well Name')
plt.ylabel('Number of Records')
plt.title('Number of Records per Well')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()

plt.show()


In [ ]:
def plot_single_well_time_series(well_data, well_name, columns, title=None, ylabel="Value"):
    """
    Plot time series of specified columns for a single well.

    Parameters:
        well_data (pd.DataFrame): DataFrame containing 'PROD_DATE' and 'WELL_NAME'
        well_name (str): Name of the well to plot (e.g., "Well_#1")
        columns (list): List of column names to plot
        title (str): Optional title for the plot
        ylabel (str): Y-axis label
    """
    # Make a copy and sort
    well_data = well_data.copy()
    well_data['PROD_DATE'] = pd.to_datetime(well_data['PROD_DATE'])
    well_data = well_data.sort_values(['WELL_NAME', 'PROD_DATE'])

    # Filter for selected well
    well_well_data = well_data[well_data['WELL_NAME'] == well_name]

    if well_well_data.empty:
        print(f"Warning: No data found for {well_name}")
        return

    # Set title
    if title is None:
        title = f"{well_name} - Time Series Plot"

    # Plot
    plt.figure(figsize=(12, 5))
    for col in columns:
        if col in well_well_data.columns:
            plt.plot(well_well_data['PROD_DATE'], well_well_data[col], label=col, linewidth=1.5, marker='o')
        else:
            print(f"Column '{col}' not found in dataset.")

    plt.title(title, fontsize=14)
    plt.xlabel('Production Date')
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_single_well_time_series(
    well_data=well_data,
    well_name='Well_#1',
    columns=[
        'BOTTOMHOLE_FLOWING_PRESSURE (PSI)',
        'ANNULUS_PRESS (PSI)',
        'WELL_HEAD_PRESSURE (PSI)'
    ],
    ylabel='Pressure (PSI)'
)

In [ ]:
plot_single_well_time_series(
    well_data=well_data,
    well_name='Well_#5',
    columns=[
        'CUMULATIVE_OIL_PROD (STB)',
        'CUMULATIVE_TOTAL_GAS_PROD (MSCF)',
        'CUMULATIVE_WATER_PROD (BBL)'
    ],
    ylabel='Cumulative Volume'
)

In [ ]:
plot_single_well_time_series(
    well_data=well_data,
    well_name='Well_#10',
    columns=[
        'DOWNHOLE_TEMPERATURE (deg F)',
        'WELL_HEAD_TEMPERATURE (deg F)'
    ],
    ylabel='Temperature (°F)'
)

In [ ]:
plot_single_well_time_series(
    well_data=well_data,
    well_name='Well_#1',
    columns=['CHOKE_SIZE (%)'],
    title="Choke Size Over Time for Well_#1",
    ylabel="Choke Size (%)"
)

In [ ]:
import matplotlib.pyplot as plt
import re

def extract_number(well_name):
    """Extract numeric index from well name for sorting."""
    match = re.search(r'\d+', str(well_name))
    return int(match.group()) if match else float('inf')

def plot_well_time_series(well_data, columns, title="Time Series Plot", ylabel="Value", 
                         show_well_numbers=True, text_position=(0.02, 0.98), text_fontsize=12):
    """
    Plots time series of specified columns for each well in subplots.
    Parameters:
        well_data (pd.DataFrame): Data containing 'PROD_DATE' and 'WELL_NAME'
        columns (list): List of column names to plot on y-axis
        title (str): Super title of the plot
        ylabel (str): Label for y-axis
        show_well_numbers (bool): Whether to show well numbers on subplots
        text_position (tuple): (x, y) position for well number text in axes coordinates
        text_fontsize (int): Font size for well number text
    """
    well_data = well_data.copy()
    well_data['PROD_DATE'] = pd.to_datetime(well_data['PROD_DATE'])
    well_data = well_data.sort_values(['WELL_NAME', 'PROD_DATE'])
    
    wells = sorted(well_data['WELL_NAME'].dropna().unique(), key=extract_number)
    n_wells = len(wells)
    
    # Define subplot grid
    n_cols = 4
    n_rows = (n_wells + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4), sharex=False)
    axes = axes.flatten()
    
    for idx, well in enumerate(wells):
        ax = axes[idx]
        well_well_data = well_data[well_data['WELL_NAME'] == well]
        
        for col in columns:
            if col in well_well_data.columns:
                ax.plot(well_well_data['PROD_DATE'], well_well_data[col], label=col, linewidth=1, marker='o')
        
        # Add bold well number at the top of each subplot (if enabled)
        if show_well_numbers:
            ax.text(text_position[0], text_position[1], f"Well #{idx+1}", 
                   transform=ax.transAxes, fontsize=text_fontsize, fontweight='bold', 
                   verticalalignment='top', bbox=dict(boxstyle='round,pad=0.3', 
                   facecolor='white', alpha=0.8))
        
        ax.set_title(f"{well}", fontsize=10)
        ax.grid(True)
        ax.tick_params(axis='x', rotation=45)
        
        if idx % n_cols == 0:
            ax.set_ylabel(ylabel)
        if idx >= (n_rows - 1) * n_cols:
            ax.set_xlabel('Production Date')
    
    # Remove unused axes
    for j in range(idx + 1, len(axes)):
        fig.delaxes(axes[j])
    
    # Shared legend
    fig.legend(columns, loc='upper center', ncol=len(columns), fontsize=12)
    
    # Layout adjustment
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.suptitle(title, fontsize=16)
    plt.show()

In [ ]:
plot_well_time_series(
    well_data=well_data,
    columns=[
        'CUMULATIVE_OIL_PROD (STB)',
        'CUMULATIVE_TOTAL_GAS_PROD (MSCF)',
        'CUMULATIVE_WATER_PROD (BBL)'
    ],
    title="Cumulative Production Profiles",
    ylabel="Cumulative Volume",
    show_well_numbers=True,        # Turn on/off well numbers
    text_position=(0.02, 0.98),   # Position as (x, y) in axes coordinates
    text_fontsize=35               # Font size for well numbers
)

In [ ]:
plot_well_time_series(
    well_data=well_data,
    columns=[
        'BOTTOMHOLE_FLOWING_PRESSURE (PSI)',
        'ANNULUS_PRESS (PSI)',
        'WELL_HEAD_PRESSURE (PSI)'
    ],
    title="Well Pressure Trends",
    ylabel="Pressure (PSI)",
    show_well_numbers=True,        # Turn on/off well numbers
    text_position=(0.02, 0.98),   # Position as (x, y) in axes coordinates
    text_fontsize=12               # Font size for well numbers
)

In [ ]:
plot_well_time_series(
    well_data=well_data,
    columns=[
        'DOWNHOLE_TEMPERATURE (deg F)',
        'WELL_HEAD_TEMPERATURE (deg F)'
    ],
    title="Well Temperature Trends",
    ylabel="Temperature (°F)",
    show_well_numbers=True,        # Turn on/off well numbers
    text_position=(0.02, 0.98),   # Position as (x, y) in axes coordinates
    text_fontsize=12               # Font size for well numbers
)

In [ ]:
plot_well_time_series(
    well_data=well_data,
    columns=['CHOKE_SIZE (%)'],
    title="Choke Size Over Time",
    ylabel="Choke Size (%)",
    show_well_numbers=True,        # Turn on/off well numbers
    text_position=(0.02, 0.98),   # Position as (x, y) in axes coordinates
    text_fontsize=12               # Font size for well numbers
)

In [ ]:
well_data.head(4)

In [ ]:
reservoir_info